<div style="background:linear-gradient(135deg,#51247a,#7a3fa0);padding:24px 28px;border-radius:10px;margin-bottom:.8rem;border-bottom:4px solid #f0a500;"><div style="font-size:2.4rem;margin-bottom:6px;">🏷️</div><div style="color:white;font-size:1.5rem;font-weight:700;">Part-of-Speech Tagger</div><div style="color:rgba(255,255,255,.82);font-size:.92rem;margin-top:4px;">Tokenise, lemmatise, and tag texts in 65+ languages<br><a href="https://ladal.edu.au/postag.html" target="_blank" style="color:#f0c060;font-size:.85rem;">&#x2192; Open the full tutorial</a></div></div>


<div style="background:#eafaf1;border-left:4px solid #27ae60;border-radius:6px;padding:14px 18px;margin:.8rem 0;font-family:sans-serif;font-size:.9rem;"><b style="color:#1a6b3a;">&#x2705; How to use this tool:</b><ol style="margin:.6rem 0 0 0;padding-left:1.3em;line-height:2.0;"><li>Upload your <code>.txt</code> files to the <b>MyTexts</b> folder (Step 2).</li><li>Set the correct <b>language model</b> in the configuration cell (Step 3).</li><li>Click <b>Kernel &#x2192; Restart &amp; Run All</b> to run the analysis.</li><li>Download the tagged Excel file and ZIP from <b>MyOutput</b> (Step 4).</li></ol></div>


<div style="background:#fff8e1;border-left:4px solid #f0a500;border-radius:6px;padding:14px 18px;margin:.8rem 0;font-family:sans-serif;font-size:.9rem;"><b style="color:#7a5000;">&#x26A1; Quick start:</b> After uploading your files and editing the configuration cell below, run all cells at once by clicking <b>Kernel &#x2192; Restart &amp; Run All</b> in the menu above. You only need to run one step manually: uploading files via the file browser panel on the left.</div>


<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.75rem;opacity:.7;text-transform:uppercase;letter-spacing:.05em;">Step 1</span><br><b style="font-size:.98rem;">&#x2699;&#xFE0F; Set up the environment</b></div>


<div style="background:#f5f5f5;border-left:3px solid #bbb;border-radius:4px;padding:5px 12px;margin-bottom:3px;font-family:sans-serif;font-size:.78rem;color:#888;">&#x1F512; <em>Runs automatically — do not edit</em></div>


In [ ]:
# Setup — loads helper functions used throughout the notebook.
# This cell runs automatically when you use Kernel > Restart & Run All.

suppressPackageStartupMessages(library(IRdisplay))

# Colour-coded feedback helpers
.ok   <- function(msg) IRdisplay::display_html(paste0(
  '<div style="background:#eafaf1;border-left:4px solid #27ae60;',
  'border-radius:5px;padding:9px 14px;margin:.35rem 0;font-family:sans-serif;">',
  '&#x2705; ', msg, '</div>'))
.warn <- function(msg) IRdisplay::display_html(paste0(
  '<div style="background:#fff8e1;border-left:4px solid #f0a500;',
  'border-radius:5px;padding:9px 14px;margin:.35rem 0;font-family:sans-serif;">',
  '&#x26A0;&#xFE0F; ', msg, '</div>'))
.err  <- function(msg) IRdisplay::display_html(paste0(
  '<div style="background:#fff0f0;border-left:4px solid #e74c3c;',
  'border-radius:5px;padding:9px 14px;margin:.35rem 0;font-family:sans-serif;">',
  '&#x274C; ', msg, '</div>'))
.info <- function(msg) IRdisplay::display_html(paste0(
  '<div style="background:#f4f0f8;border-left:4px solid #51247a;',
  'border-radius:5px;padding:9px 14px;margin:.35rem 0;font-family:sans-serif;">',
  '&#x2139;&#xFE0F; ', msg, '</div>'))
.prog <- function(label, pct) IRdisplay::display_html(paste0(
  '<div style="font-family:sans-serif;font-size:.85rem;margin:.3rem 0;">',
  '<b style="color:#51247a;">', label, '</b>',
  '<div style="background:#e8e4f0;border-radius:4px;height:7px;margin-top:3px;">',
  '<div style="background:#51247a;width:', round(pct), '%;height:7px;',
  'border-radius:4px;"></div></div></div>'))

# Load plain-text files from a folder
load_texts <- function(folder = "notebooks/MyTexts") {
  # Create folder if it does not exist (git does not track empty dirs)
  dir.create(folder, showWarnings = FALSE, recursive = TRUE)
  files <- list.files(folder, pattern = "(?i)\\.txt$",
                      full.names = TRUE, recursive = FALSE)
  if (length(files) == 0)
    stop(
    "No .txt files found in '", folder, "'\n\n",
    "Please make sure:\n",
    "  1. You opened the correct folder in the file browser (left panel)\n",
    "  2. You dragged .txt files into that folder\n",
    "  3. Files have a .txt extension (not .docx, .pdf, etc.)\n",
    "  4. You ran Kernel > Restart & Run All AFTER uploading")
  texts <- vapply(files, function(f)
    paste(readLines(f, warn = FALSE, encoding = "UTF-8"), collapse = " "),
    character(1))
  names(texts) <- tools::file_path_sans_ext(basename(files))
  texts
}

# Save named character vector as .txt files
save_texts <- function(texts, folder = "notebooks/MyOutput") {
  dir.create(folder, showWarnings = FALSE, recursive = TRUE)
  for (nm in names(texts))
    writeLines(texts[[nm]], file.path(folder, paste0(nm, ".txt")))
}

# Save data frame as Excel
save_excel <- function(df, filename, folder = "notebooks/MyOutput") {
  dir.create(folder, showWarnings = FALSE, recursive = TRUE)
  writexl::write_xlsx(as.data.frame(df), file.path(folder, filename))
}

.ok("Setup complete.")


<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.75rem;opacity:.7;text-transform:uppercase;letter-spacing:.05em;">Step 2</span><br><b style="font-size:.98rem;">&#x1F4C2; Upload your texts to the MyTexts folder</b></div>


<div style="background:#f4f0f8;border:2px dashed #51247a;border-radius:8px;padding:18px 22px;margin:.6rem 0;font-family:sans-serif;"><b style="color:#51247a;font-size:.95rem;">&#x1F4C2; Upload your files to <code>MyTexts</code></b><ol style="margin:.6rem 0 0 0;padding-left:1.3em;font-size:.9rem;line-height:1.9;"><li>Open the <b>file browser panel</b> on the left of the screen (click the folder icon if it is not visible).</li><li>Double-click the <b><code>MyTexts</code></b> folder.</li><li><b>Drag and drop</b> your <code>.txt</code> files into that folder.</li><li>Then click <b>Kernel &#x2192; Restart &amp; Run All</b> to run the notebook.</li></ol><p style="margin:.5rem 0 0 0;font-size:.82rem;color:#888;">Only <code>.txt</code> files are accepted.</p></div>


<div style="background:#e8f4fd;border-left:4px solid #4085C6;border-radius:4px;padding:7px 14px;margin-bottom:3px;font-family:sans-serif;font-size:.82rem;color:#2a5ea8;">&#x270F;&#xFE0F; <b>Edit the values below</b>, then run this cell (Shift+Enter or click &#x25B6; Run in the toolbar).</div>


In [ ]:
# ── CONFIGURATION ─────────────────────────────────────────────────────
# Edit the language model below, then run this cell (Shift+Enter).
#
# Common models:
#   "english-ewt"       English (recommended)
#   "english-gum"       English (alternative)
#   "german-gsd"        German
#   "french-gsd"        French
#   "spanish-ancora"    Spanish
#   "italian-isdt"      Italian
#   "dutch-alpino"      Dutch
#   "portuguese-bosque" Portuguese
#   "russian-gsd"       Russian
#   "chinese-gsd"       Chinese
#   "arabic-padt"       Arabic
#   "japanese-gsd"      Japanese
# Full list: https://ladal.edu.au/postag.html

language_model <- "english-ewt"


<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.75rem;opacity:.7;text-transform:uppercase;letter-spacing:.05em;">Step 3</span><br><b style="font-size:.98rem;">&#x1F4CA; Run the analysis</b></div>


<div style="background:#f5f5f5;border-left:3px solid #bbb;border-radius:4px;padding:5px 12px;margin-bottom:3px;font-family:sans-serif;font-size:.78rem;color:#888;">&#x1F512; <em>Runs automatically — do not edit</em></div>


In [ ]:
# Part-of-speech tagging — runs automatically.
suppressPackageStartupMessages(library(udpipe))
suppressPackageStartupMessages(library(writexl))
suppressPackageStartupMessages(library(dplyr))
suppressPackageStartupMessages(library(ggplot2))
suppressPackageStartupMessages(library(zip))

texts <- tryCatch(load_texts("notebooks/MyTexts"),
  error = function(e) { .err(conditionMessage(e)); NULL })
if (!is.null(texts)) {
  .info(paste0("Loaded ", length(texts), " file(s)."))
  .prog("Loading language model...", 15)
  model_dir <- file.path(Sys.getenv("HOME", "/home/jovyan"), "udpipe-models")
  dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)
  existing <- list.files(model_dir,
    pattern = paste0("^", language_model, ".*\\.udpipe$"),
    full.names = TRUE, recursive = TRUE)
  if (length(existing) > 0) {
    model_path <- existing[1]
    .ok(paste("Using cached model:", basename(model_path)))
  } else {
    .info("Downloading language model (30-60 seconds)...")
    dl <- udpipe_download_model(language = language_model,
                                model_dir = model_dir)
    if (isTRUE(dl$download_failed))
      stop("Model download failed. Check your internet connection.")
    model_path <- dl$file_model
    .ok(paste("Downloaded:", basename(model_path)))
  }
  model <- udpipe_load_model(model_path)
  tagged_list <- lapply(seq_along(texts), function(i) {
    nm <- names(texts)[i]
    .prog(paste0("Tagging: ", nm, " (", i, "/", length(texts), ")"),
          20 + 65 * (i / length(texts)))
    as.data.frame(udpipe_annotate(model, x = texts[[nm]], doc_id = nm))
  })
  tagged_df <- dplyr::bind_rows(tagged_list) |>
    dplyr::select(doc_id, sentence_id, token_id, token, lemma,
                  upos, xpos, dep_rel, head_token_id) |>
    dplyr::rename(Document = doc_id, Sentence = sentence_id,
                  TokenID = token_id, Token = token, Lemma = lemma,
                  UPOS = upos, XPOS = xpos, DepRel = dep_rel,
                  HeadTokenID = head_token_id)
  .prog("Plotting POS summary...", 88)
  pos_counts <- tagged_df |>
    dplyr::filter(!is.na(UPOS), !UPOS %in% c("PUNCT", "SPACE")) |>
    dplyr::count(UPOS, sort = TRUE)
  p <- ggplot(pos_counts, aes(x = reorder(UPOS, n), y = n)) +
    geom_col(fill = "#51247a", width = .7) +
    geom_text(aes(label = format(n, big.mark = ",")),
              hjust = -0.1, size = 3.5, colour = "gray30") +
    coord_flip() +
    scale_y_continuous(expand = expansion(mult = c(0, .15))) +
    theme_bw(base_size = 12) +
    labs(title = "Token count by POS category",
         x = "Universal POS tag", y = "Count")
  print(p)
  save_excel(tagged_df, "pos_tagged.xlsx")
  dir.create("notebooks/MyOutput", showWarnings = FALSE, recursive = TRUE)
  ggsave("notebooks/MyOutput/pos_summary.png",
         bg = "white", width = 7, height = 5, dpi = 200)
  tmp <- tempfile(); dir.create(tmp)
  for (doc in unique(tagged_df$Document)) {
    df <- dplyr::filter(tagged_df, Document == doc,
                        !is.na(Token), !is.na(UPOS))
    writeLines(paste(paste0(df$Token, "_", df$UPOS), collapse = " "),
               file.path(tmp, paste0(doc, "_postag.txt")))
  }
  zip::zip("notebooks/MyOutput/pos_tagged_texts.zip",
           files = list.files(tmp, full.names = TRUE), mode = "cherry-pick")
  .ok(paste0("Tagged <b>", nrow(tagged_df), " tokens</b>. ",
             "Saved pos_tagged.xlsx, pos_summary.png, pos_tagged_texts.zip."))
}


<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.75rem;opacity:.7;text-transform:uppercase;letter-spacing:.05em;">Step 4</span><br><b style="font-size:.98rem;">&#x2B07;&#xFE0F; Download your results</b></div>


<div style="background:#eafaf1;border-left:4px solid #27ae60;border-radius:6px;padding:12px 18px;margin:.6rem 0;font-family:sans-serif;font-size:.9rem;"><b style="color:#1a6b3a;">&#x2B07;&#xFE0F; Download your results</b><br>Open the <b>file browser</b> on the left, double-click <b>MyOutput</b>, then <b>right-click</b> any file and choose <b>Download</b>.</div>


---

### Citation

Schweinberger, Martin. (2025). *LADAL Part-of-Speech Tagger*. Brisbane: The University of Queensland. <https://ladal.edu.au/tools.html>

```bibtex
@misc{schweinberger2025postagger,
  author = {Schweinberger, Martin},
  title  = {LADAL Part-of-Speech Tagger},
  year   = {2025},
  organization = {The University of Queensland},
  url    = {https://ladal.edu.au/tools.html}
}
```


In [ ]:
sessionInfo()